In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Fausto\AppData\Local\Python\pythoncore-3.14-64\python.exe
3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [2]:
import duckdb
print("duckdb ok")

duckdb ok


In [26]:
import duckdb

con = duckdb.connect()

path = r"C:/Users/Fausto/Downloads/train-part-*_extracted/train-part-*/*.parquet"

con.execute(f"""
CREATE OR REPLACE VIEW notif AS
SELECT *
FROM read_parquet('{path}');
""")

print(con.execute("SELECT COUNT(*) FROM notif").fetchdf())
print(con.execute("SELECT AVG(CAST(session_end_completed AS INT)) FROM notif").fetchdf())

   count_star()
0      87665839
   avg(CAST(session_end_completed AS INTEGER))
0                                     0.143826


1) How big is this dataset?

In [27]:
con.execute(f"""
SELECT COUNT(*) AS n_rows
FROM read_parquet('{path}')
""").df()

,n_rows
0,87665839


2) What is the global sucess rate (reward)?

In [28]:
con.execute(f"""
SELECT AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
""").df()

,reward_rate
0,0.143826


In [29]:
con.execute(f"""
SELECT
  regexp_extract(filename, 'train-part-\\d+', 0) AS part,
  COUNT(*) AS n,
  AVG(CAST(session_end_completed AS INT)) AS reward_rate
FROM read_parquet('{path}', filename=true)
GROUP BY 1
ORDER BY 1
""").df()

,part,n,reward_rate
0,train-part-1,25613243,0.178333
1,train-part-2,25501113,0.178502
2,train-part-3,36551483,0.095452


3) How often occurs every template + how good does it work?

In [30]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate
0,C,2523858,0.412298
1,A,3472696,0.268958
2,L,9062239,0.131680
3,G,9058068,0.131586
4,K,9129646,0.131572
5,D,9054749,0.130969
6,J,9062686,0.130362
7,E,9060589,0.129533
8,F,9057400,0.128914
9,H,9124283,0.128724


4) Does it differ per language?

In [31]:
con.execute(f"""
SELECT
  ui_language,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
GROUP BY ui_language
ORDER BY n DESC
""").df()

,ui_language,n,reward_rate
0,en,35782594,0.162151
1,es,21291632,0.122337
2,pt,8296662,0.120473
3,ru,3957298,0.142532
4,fr,3364800,0.149757
5,de,2490364,0.195564
6,ar,1730233,0.063945
7,zs,1598546,0.099429
8,it,1477620,0.153406
9,vi,1389414,0.144972


5) Important for bandit: how many templates are eligible per event?

In [32]:
con.execute(f"""
SELECT
  AVG(LEN(eligible_templates)) AS avg_eligible,
  MIN(LEN(eligible_templates)) AS min_eligible,
  MAX(LEN(eligible_templates)) AS max_eligible
FROM read_parquet('{path}')
""").df()

,avg_eligible,min_eligible,max_eligible
0,9.147085,1,10


How often is C available?

In [33]:
con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN 'C' IN eligible_templates THEN 1 ELSE 0 END) AS c_available
FROM read_parquet('{path}')
""").df()

,total,c_available
0,87665839,2523858.0


What if we always choose C when C is available, 
and otherwise the best of the rest?

So firstly we must know:
What is the best template without C?

In [34]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
WHERE 'C' NOT IN eligible_templates
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate
0,A,3472696,0.268958
1,L,9062239,0.131680
2,G,9058068,0.131586
3,K,9129646,0.131572
4,D,9054749,0.130969
5,J,9062686,0.130362
6,E,9060589,0.129533
7,F,9057400,0.128914
8,H,9124283,0.128724
9,B,9059625,0.128534


There is a clear hierarchy:
C-> best overall(41.2%)
A -> best if C is not available (26.9%)
Rest -> all around 13%

This is no subtle difference.
A strong rule-based policy would be:
If C available -> choose C
Else -> choose A

1) How often would our new policy choose C vs A?

In [35]:
con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN 'C' IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_C,
  SUM(CASE WHEN 'C' NOT IN eligible_templates AND 'A' IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_A,
  SUM(CASE WHEN 'C' NOT IN eligible_templates AND 'A' NOT IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_neither
FROM read_parquet('{path}')
""").df()

,total,policy_choose_C,policy_choose_A,policy_choose_neither
0,87665839,2523858.0,34402243.0,50739738.0


2) Off-policy estimate: reward of policy “C else A”

In [36]:
con.execute(f"""
WITH data AS (
  SELECT
    session_end_completed,
    selected_template,
    CASE 
      WHEN 'C' IN eligible_templates THEN 'C'
      ELSE 'A'
    END AS policy_template
  FROM read_parquet('{path}')
)
SELECT
  COUNT(*) AS total_events,
  SUM(CASE WHEN selected_template = policy_template THEN 1 ELSE 0 END) AS matched_events,
  AVG(CASE WHEN selected_template = policy_template THEN CASE WHEN session_end_completed THEN 1 ELSE 0 END END) AS estimated_policy_reward
FROM data
""").df()

,total_events,matched_events,estimated_policy_reward
0,87665839,5996554.0,0.329288


What do we see?
Total events
87,665,839
Matched events
5,996,554 (useful for evaluation)
Estimated policy reward
0.329 (32.9%)
Comparison with current logging policy(=In the historical data, Duolingo selected a template at random from the eligible pool, with equal probability for each template. That random decision process is what generated the dataset — that’s the logging policy)
Logging policy reward was:
14.4%
Our simple new rule:
If C available -> C
Else -> A
32.9%

A matched event is:
An event where the logging policy chose by chance exactly the same template as your new policy would have chosen.
Why do we only use matched events?

!!!Because we only know the reward of what was actually shown!!!

For example:
Suppose:
Eligible = {C, D, E}
Logging chose D
Our new policy would have chosen C
Then we know:
Reward of D -> yes
Reward of C -> no (counterfactual unknown)
So we can't use that event to evaluate C.

This compares C vs A in exactly the same context/set/pool
It removes a big part of the selection bias.

In [37]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
WHERE 'C' IN eligible_templates
  AND 'A' IN eligible_templates
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate


What does this mean in terms of content?

!!!C and A apparently never occur in the same eligible pool!!!

So:
!!!C is shown in a completely different segment than A,
this confirms selection bias!!!
You can't compare C and A within the same context.

The rule:
If C available -> C
Else -> A
actually works because:
C-segment = high engagement users
A-segment = other users
So your policy is actually:
If user in C-segment -> choose C
Otherwise -> choose A
But that segment difference is already incorporated in the data.

1) Which eligible pools occur the most?

In [38]:
con.execute(f"""
SELECT
  eligible_templates,
  COUNT(*) AS n
FROM read_parquet('{path}')
GROUP BY eligible_templates
ORDER BY n DESC
LIMIT 20
""").df()

,eligible_templates,n
0,"[G, E, B, K, H, J, L, F, D]",25576117
1,"[K, H, G, E, B, J, L, F, D]",25112950
2,"[G, E, B, A, K, H, J, L, F, D]",19498003
3,"[K, H, G, E, B, J, L, F, D, A]",14766763
4,[C],2523858
5,"[A, K, H]",79182
6,"[K, H, A]",58291
7,"[K, H]",50671
8,"[G, E, B, A]",4


2) Within every pool: reward per selected_template (and top/best template per pool)

In [39]:
con.execute(f"""
WITH stats AS (
  SELECT
    eligible_templates,
    selected_template,
    COUNT(*) AS n,
    AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
  FROM read_parquet('{path}')
  GROUP BY eligible_templates, selected_template
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY eligible_templates
      ORDER BY reward_rate DESC
    ) AS rnk
  FROM stats
  WHERE n >= 5000
)
SELECT
  eligible_templates,
  selected_template AS best_template,
  n,
  reward_rate
FROM ranked
WHERE rnk = 1
ORDER BY n DESC
LIMIT 50
""").df()

,eligible_templates,best_template,n,reward_rate
0,"[G, E, B, K, H, J, L, F, D]",L,2842337,0.050306
1,"[K, H, G, E, B, J, L, F, D]",L,2792022,0.044578
2,[C],C,2523858,0.412298
3,"[G, E, B, A, K, H, J, L, F, D]",G,1948752,0.272927
4,"[K, H, G, E, B, J, L, F, D, A]",G,1475787,0.268473
5,"[A, K, H]",A,26401,0.454301
6,"[K, H]",H,25172,0.099992
7,"[K, H, A]",A,19507,0.448865


What do we see here?
There are a few big eligible pools (million events).
The best template is not always C within these pools.
In some pools wins L.
In little pools (such as [A, K, H]) wins A with a high reward rate.
C is actually in its own pool= [C].
That means:
Templates live in different worlds.
Reward rate is strongly dependent of the eligible pool.

Logistic Model:

STEP1 : load the data (fast + safe) with DuckDB in Python

In [43]:
import duckdb
import pandas as pd

# 1) Connect
con = duckdb.connect()

# 2) Zet hier je pad naar de parquet files (training of test)
# Voorbeeld: "data/training/*.parquet" of "/path/to/training/*.parquet"
PARQUET_GLOB = r"C:/Users/Fausto/Downloads/train-part-*_extracted/train-part-*/*.parquet"

# 3) Maak een view (handig om later queries te hergebruiken)
con.execute(f"""
CREATE OR REPLACE VIEW notif AS
SELECT *
FROM read_parquet('{PARQUET_GLOB}');
""")

# 4) Snelle sanity checks
print("Row count (approx query):")
print(con.execute("SELECT COUNT(*) AS n FROM notif").fetchdf())

print("\nColumns + types:")
print(con.execute("DESCRIBE notif").fetchdf())

print("\nOverall reward rate:")
print(con.execute("""
SELECT AVG(CAST(session_end_completed AS INTEGER)) AS reward_rate
FROM notif
""").fetchdf())

print("\nExample rows:")
print(con.execute("""
SELECT datetime, ui_language, eligible_templates, history, selected_template, session_end_completed
FROM notif
LIMIT 3
""").fetchdf())

Row count (approx query):
          n
0  87665839

Columns + types:
             column_name                                 column_type null  \
0               datetime                                      DOUBLE  YES   
1            ui_language                                     VARCHAR  YES   
2     eligible_templates                                   VARCHAR[]  YES   
3                history  STRUCT("template" VARCHAR, n_days FLOAT)[]  YES   
4      selected_template                                     VARCHAR  YES   
5  session_end_completed                                     BOOLEAN  YES   

    key default extra  
0  None    None  None  
1  None    None  None  
2  None    None  None  
3  None    None  None  
4  None    None  None  
5  None    None  None  

Overall reward rate:
   reward_rate
0     0.143826

Example rows:
   datetime ui_language              eligible_templates  \
0  0.153461          en  [G, E, B, A, K, H, J, L, F, D]   
1  2.827303          es  [G, E, B, A, K

Step 2: drawing a training sample + prepare features
We want:
Only real decision points (len(eligible_templates) >= 2)
A sample of ±1 miljoen rows (for speed)

In [44]:
query = """
SELECT
    ui_language,
    selected_template,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward
FROM notif
WHERE array_length(eligible_templates) >= 2
USING SAMPLE 1000000 ROWS
"""

df = con.execute(query).df()

print(df.shape)
print(df.head())
print("Reward rate in sample:", df['reward'].mean())

(971244, 4)
  ui_language selected_template  n_eligible  reward
0          pt                 J           9       0
1          cs                 L          10       1
2          tr                 E           9       0
3          en                 L           9       0
4          en                 F           9       0
Reward rate in sample: 0.13624279789630617


Step 3 — Logistic Regression training

In [6]:
!pip install scikit-learn

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [45]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

X = df[['ui_language', 'selected_template', 'n_eligible']]
y = df['reward']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'),
         ['ui_language', 'selected_template']),
        ('num', 'passthrough', ['n_eligible'])
    ]
)

model = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)

print("AUC:", auc)

AUC: 0.7424332785794168


Wat does 0.74 AUC mean concrete?
0.5 = random guessing
0.6 = poor signal
0.7 = solidly predictive
0.8+ = strong model
So:
There is a clear structure in the data.
!!!Template + language really matters!!!

Stap 4A — Check template ranking according to model

In [46]:
import numpy as np

# Extract coefficients
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefs = model.named_steps['clf'].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
})

# Kijk enkel naar template effecten
template_effects = coef_df[coef_df["feature"].str.contains("selected_template")]

template_effects.sort_values("coef", ascending=False).head(10)

,feature,coef
23,cat__selected_template_A,-1.526394
31,cat__selected_template_K,-1.583652
32,cat__selected_template_L,-1.606074
26,cat__selected_template_E,-1.610331
29,cat__selected_template_H,-1.613078
28,cat__selected_template_G,-1.615483
30,cat__selected_template_J,-1.616740
25,cat__selected_template_D,-1.618800
27,cat__selected_template_F,-1.619022
24,cat__selected_template_B,-1.648990


What we want to test now...
We want to test:
If we choose the template with the highest predicted probability within every eligible pool,
do we receive a higher reward than random?

That's the core.
But here is the problem
In your current dataset you have only:
selected_template

!!!Because we only know the reward of what was actually shown.
You don't know what the reward would have been for the other templates in that pool.
So we can't simulate perfectly.!!!

But...
Because logging policy was uniform random (implies selection bias)
we can do a correct off-policy evaluation.

Goal:

Simulate what would have happened if we choose the template with the highest predicted probability within every pool.

Stap 4 — Model-based policy simulation

Stap 4A — New sample with eligible_templates

In [47]:
query = """
SELECT
    ui_language,
    selected_template,
    eligible_templates,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward
FROM notif
WHERE array_length(eligible_templates) >= 2
USING SAMPLE 300000 ROWS
"""

df_policy = con.execute(query).df()
print(df_policy.shape)

(291314, 5)


Stap 4B — Policy simulation

In [48]:
import pandas as pd
import numpy as np

# df_policy moet deze kolommen hebben:
# ['ui_language', 'selected_template', 'eligible_templates', 'n_eligible', 'reward']

# 1) Geef elke rij een id zodat we later kunnen groeperen
df_policy = df_policy.reset_index(drop=True)
df_policy['row_id'] = np.arange(len(df_policy))

# 2) Explode eligible_templates -> long format
long = df_policy[['row_id', 'ui_language', 'n_eligible', 'reward', 'selected_template', 'eligible_templates']].explode('eligible_templates')
long = long.rename(columns={'eligible_templates': 'cand_template'})

# 3) Predict in batch: we zetten cand_template tijdelijk in de kolomnaam die het model verwacht
X_long = long[['ui_language', 'n_eligible']].copy()
X_long['selected_template'] = long['cand_template'].astype(str)

# batch predict
long['p_hat'] = model.predict_proba(X_long)[:, 1]

# 4) Kies beste template per row_id
best_idx = long.groupby('row_id')['p_hat'].idxmax()
chosen = long.loc[best_idx, ['row_id', 'cand_template']].rename(columns={'cand_template': 'model_choice'})

# 5) Join terug op df_policy
df_eval = df_policy.merge(chosen, on='row_id', how='left')

# 6) Matched evaluation
matched = df_eval[df_eval['model_choice'] == df_eval['selected_template']]

matched_fraction = len(matched) / len(df_eval)
reward_matched = matched['reward'].mean()
reward_baseline = df_eval['reward'].mean()

print("Matched fraction:", matched_fraction)
print("Reward in matched cases:", reward_matched)
print("Random baseline reward:", reward_baseline)

# Uplift (absolute en relatief)
print("Absolute uplift:", reward_matched - reward_baseline)
print("Relative uplift:", (reward_matched / reward_baseline) - 1)

Matched fraction: 0.10651509805462979
Reward in matched cases: 0.13482359241231112
Random baseline reward: 0.1401825624187601
Absolute uplift: -0.005358970006448971
Relative uplift: -0.03822850655590382


Stap 5 — IPS evaluatie

In [49]:
# IPS evaluation for deterministic policy "choose model_choice"
# Assumption: logging was uniform random over eligible_templates
# => b(a|t) = 1 / n_eligible  -> weight = n_eligible

df_eval['match'] = (df_eval['model_choice'] == df_eval['selected_template']).astype(int)
df_eval['w'] = df_eval['n_eligible']  # 1 / (1/n_eligible)

ips_estimate = (df_eval['match'] * df_eval['reward'] * df_eval['w']).mean()

baseline = df_eval['reward'].mean()

print("IPS estimated reward of model-policy:", ips_estimate)
print("Random baseline reward:", baseline)
print("Absolute uplift:", ips_estimate - baseline)
print("Relative uplift:", ips_estimate / baseline - 1)

IPS estimated reward of model-policy: 0.1401825624187601
Random baseline reward: 0.1401825624187601
Absolute uplift: 0.0
Relative uplift: 0.0


Let's add recency (= how many days ago a template was last shown to the user) and after that again:

model training

policy simulation (vectorized)

IPS evaluation

1) Sample with recency + eligible_templates

In [50]:
query = """
SELECT
    ui_language,
    selected_template,
    eligible_templates,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward,

    (
      SELECT min(h.n_days)
      FROM unnest(history) AS u(h)
      WHERE h.template = selected_template
    ) AS days_since_last_seen

FROM notif
WHERE array_length(eligible_templates) >= 2
USING SAMPLE 300000 ROWS
"""
df = con.execute(query).df()

print(df.shape)
print("Missing recency:", df['days_since_last_seen'].isna().mean())
print(df.head())

(291203, 6)
Missing recency: 0.5080785568829992
  ui_language selected_template              eligible_templates  n_eligible  \
0          es                 J  [G, E, B, A, K, H, J, L, F, D]          10   
1          es                 B     [K, H, G, E, B, J, L, F, D]           9   
2          pt                 K     [G, E, B, K, H, J, L, F, D]           9   
3          en                 B  [K, H, G, E, B, J, L, F, D, A]          10   
4          en                 F     [G, E, B, K, H, J, L, F, D]           9   

   reward  days_since_last_seen  
0       1                   NaN  
1       0                   NaN  
2       0              7.975042  
3       0                   NaN  
4       0             17.967855  


Next stap (Stap 2) — preparing recency effect

In [51]:
df['days_since_last_seen'] = df['days_since_last_seen'].fillna(999).astype(float)

print(df['days_since_last_seen'].describe())
print("Share 'never' (999):", (df['days_since_last_seen'] == 999).mean())

count    291203.000000
mean        511.314791
std         495.654075
min           0.003463
25%           5.029721
50%         999.000000
75%         999.000000
max         999.000000
Name: days_since_last_seen, dtype: float64
Share 'never' (999): 0.5080785568829992


After that: again logistic regression training (with recency effect)

In [52]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

X = df[['ui_language', 'selected_template', 'n_eligible', 'days_since_last_seen']]
y = df['reward']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'),
         ['ui_language', 'selected_template']),
        ('num', 'passthrough', ['n_eligible', 'days_since_last_seen'])
    ]
)

model_rec = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=1000))
])

model_rec.fit(X_train, y_train)

y_pred = model_rec.predict_proba(X_test)[:, 1]
print("AUC with recency:", roc_auc_score(y_test, y_pred))

AUC with recency: 0.7599904743491368


c:\Users\Fausto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [53]:
query_long = """
WITH base AS (
  SELECT
    row_number() OVER () - 1 AS row_id,
    ui_language,
    selected_template,
    eligible_templates,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward,
    history
  FROM notif
  WHERE array_length(eligible_templates) >= 2
  USING SAMPLE 200000 ROWS
),
cand AS (
  SELECT
    b.row_id,
    b.ui_language,
    b.n_eligible,
    b.reward,
    b.selected_template,
    t AS cand_template,
    (
      SELECT min(h.n_days)
      FROM unnest(b.history) AS u(h)
      WHERE h.template = t
    ) AS days_since_last_seen_cand
  FROM base b
  CROSS JOIN unnest(b.eligible_templates) AS u(t)
)
SELECT *
FROM cand;
"""

long = con.execute(query_long).df()
print(long.shape)
print("Missing candidate recency:", long['days_since_last_seen_cand'].isna().mean())

(1824161, 7)
Missing candidate recency: 0.5087648513480992


It means:
Half of candidate templates are “new” (or unseen recently)
That novelty signal exists but even using it, greedy didn’t improve IPS reward